# Evaluate registered BioASQ retrieval datasets

This notebook registers an immutable retrieval pipeline and evaluates it on every valid dataset below `DATASETS_ROOT`. Only the latest registered version of each dataset name is evaluated. Existing results for the same pipeline and dataset version are loaded instead of recomputed.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets sentence-transformers


In [ ]:
import logging

import pandas as pd
import torch
from IPython.display import display

from src.evaluate import (
    SimilarityMetric,
    evaluate,
    register_pipeline,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("retrieval-test")


## Configuration

The pipeline identity contains the model, similarity function, evaluator metrics, batch sizes, and model or evaluator keyword arguments. `DEVICE` is runtime information and does not change the pipeline identity.

In [ ]:
DATASETS_ROOT = "/content/drive/MyDrive/Retreaval/data"
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SIMILARITY_METRIC = SimilarityMetric.COSINE
BATCH_SIZE = 64
CORPUS_CHUNK_SIZE = 10_000
METRIC_CONFIG = {
    "mrr_at_k": (10,),
    "ndcg_at_k": (10,),
    "accuracy_at_k": (1, 3, 5, 10, 100),
    "precision_recall_at_k": (1, 3, 5, 10, 100),
    "map_at_k": (100,),
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", DEVICE)


## Register the pipeline

Repeated registration of the same configuration returns the same `pipeline_id`. Changing the model, scorer, metrics, or another stored parameter creates a different pipeline.

In [ ]:
pipeline_id = register_pipeline(
    model_name=MODEL_NAME,
    similarity_metric=SIMILARITY_METRIC,
    registry_db_path=REGISTRY_DB_PATH,
    batch_size=BATCH_SIZE,
    corpus_chunk_size=CORPUS_CHUNK_SIZE,
    metric_config=METRIC_CONFIG,
    show_progress_bar=True,
)
print(f"Pipeline: {pipeline_id}")


## Evaluate every dataset

`evaluate()` discovers all saved datasets directly below `DATASETS_ROOT`, registers their current content, checks the latest version in `datasets.sqlite`, and writes missing results to `results.sqlite`. The uniqueness key is `(pipeline_id, dataset_id)`, so different scoring configurations of the same model remain separate.

In [ ]:
outcomes = evaluate(
    pipeline_id=pipeline_id,
    datasets_root=DATASETS_ROOT,
    registry_db_path=REGISTRY_DB_PATH,
    results_db_path=RESULTS_DB_PATH,
    device=DEVICE,
)


In [ ]:
records = []
for outcome in outcomes:
    record = {
        "dataset": outcome.dataset_name,
        "version": outcome.dataset_version,
        "status": outcome.status.value,
        "dataset_id": outcome.dataset_id,
        "result_id": outcome.result_id,
    }
    if outcome.metrics is not None:
        record.update(outcome.metrics)
    records.append(record)

results_table = pd.DataFrame(records).sort_values(
    ["dataset", "version"]
)
identifier_columns = {
    "dataset",
    "version",
    "status",
    "dataset_id",
    "result_id",
}
metric_columns = [
    column
    for column in results_table.columns
    if column not in identifier_columns
]
display(results_table.style.format(
    {column: "{:.4f}" for column in metric_columns},
    na_rep="",
))
